# Secure coding practices

# Coding problem, 5.5.b: Secure-coding configuration auditor

**Skill:** Mitigate Risk · **Maps to:** 5.5.A, 5.5.B · **Language:** Python 3 (pytest)

## Problem
Build a lightweight policy-as-code auditor: given an application's configuration, flag every
secure-coding practice it violates, exactly the checks a code reviewer or CI gate would run.

## Function contract
Implement `solve(config: dict) -> list` in the code editor (see the Reference section below).

## Input
`config`: `dict` of settings. Any key may be missing.

## Output
Sorted `list[str]` of violated rule names.

## Rule set
| condition | violation |
|---|---|
| `debug_mode is True` | `"debug-mode-enabled"` |
| `db_query_style == "concatenated"` | `"non-parameterized-query"` |
| `secrets_in_code is True` | `"hardcoded-secret"` |
| `error_verbosity == "verbose"` | `"verbose-error-messages"` |
| `dependencies_pinned is False` | `"unpinned-dependencies"` |
| `runs_as == "root"` | `"excessive-privilege"` |
| `code_review_required is False` | `"no-code-review"` |

## Examples
```
{"debug_mode": True}                         ->  ["debug-mode-enabled"]
{"db_query_style": "parameterized"}          ->  []
{"runs_as": "root", "secrets_in_code": True} ->  ["excessive-privilege", "hardcoded-secret"]
```

## Constraints
- Deterministic, offline, standard library only.
- A missing key never triggers a violation.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(config: dict) -> list
```

### Input (`config`)
`dict` of application settings. Any key may be missing (missing keys never trigger a violation).
Recognized keys:
- `"debug_mode"`: `bool`
- `"db_query_style"`: `str` (`"parameterized"` or `"concatenated"`)
- `"secrets_in_code"`: `bool`
- `"error_verbosity"`: `str` (`"generic"` or `"verbose"`)
- `"dependencies_pinned"`: `bool`
- `"runs_as"`: `str` (e.g. `"app_user"` or `"root"`)
- `"code_review_required"`: `bool`

### Output (`result`)
Sorted `list[str]` of violated rule names. Only rules that fail appear.

### Rule set
| condition | violation |
|---|---|
| `debug_mode is True` | `"debug-mode-enabled"` |
| `db_query_style == "concatenated"` | `"non-parameterized-query"` |
| `secrets_in_code is True` | `"hardcoded-secret"` |
| `error_verbosity == "verbose"` | `"verbose-error-messages"` |
| `dependencies_pinned is False` | `"unpinned-dependencies"` |
| `runs_as == "root"` | `"excessive-privilege"` |
| `code_review_required is False` | `"no-code-review"` |

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- A missing key never counts as a violation (only an explicit disqualifying value does).

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 5.5.b. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict, List


def solve(config: Dict[str, object]) -> List[str]:
    """Audit an application configuration against the secure-coding rule set.

    Args:
        config: dict of settings, e.g. {"debug_mode": bool, "db_query_style": str,
            "secrets_in_code": bool, "error_verbosity": str, "dependencies_pinned": bool,
            "runs_as": str, "code_review_required": bool}. Any key may be missing.
    Returns:
        sorted list[str] of violated rule names. See the Reference section below for the exact rule set.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_all_secure_no_violations():
    config = {
        "debug_mode": False,
        "db_query_style": "parameterized",
        "secrets_in_code": False,
        "error_verbosity": "generic",
        "dependencies_pinned": True,
        "runs_as": "app_user",
        "code_review_required": True,
    }
    assert impl.solve(config) == []


def test_debug_mode_flagged():
    config = {"debug_mode": True}
    assert impl.solve(config) == ["debug-mode-enabled"]


def test_concatenated_query_flagged():
    config = {"db_query_style": "concatenated"}
    assert impl.solve(config) == ["non-parameterized-query"]


def test_hardcoded_secret_flagged():
    config = {"secrets_in_code": True}
    assert impl.solve(config) == ["hardcoded-secret"]


def test_multiple_violations_sorted():
    config = {
        "debug_mode": True,
        "runs_as": "root",
        "secrets_in_code": True,
    }
    assert impl.solve(config) == ["debug-mode-enabled", "excessive-privilege", "hardcoded-secret"]


# ---- hidden / edge cases -------------------------------------------------
def test_verbose_errors_flagged():
    config = {"error_verbosity": "verbose"}
    assert impl.solve(config) == ["verbose-error-messages"]


def test_unpinned_dependencies_flagged():
    config = {"dependencies_pinned": False}
    assert impl.solve(config) == ["unpinned-dependencies"]


def test_no_code_review_flagged():
    config = {"code_review_required": False}
    assert impl.solve(config) == ["no-code-review"]


def test_missing_keys_treated_as_not_violating():
    assert impl.solve({}) == []


def test_all_violations_at_once():
    config = {
        "debug_mode": True,
        "db_query_style": "concatenated",
        "secrets_in_code": True,
        "error_verbosity": "verbose",
        "dependencies_pinned": False,
        "runs_as": "root",
        "code_review_required": False,
    }
    assert impl.solve(config) == [
        "debug-mode-enabled",
        "excessive-privilege",
        "hardcoded-secret",
        "no-code-review",
        "non-parameterized-query",
        "unpinned-dependencies",
        "verbose-error-messages",
    ]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
